# 04 — Agent 2 : fusion des annotations en portions

L'Agent 2 est déterministe.

Il reçoit plusieurs annotations qui peuvent se chevaucher :

```text
forte_pente : 1000 ───────── 1500
lacets      :      1250 ───────── 1750
```

et construit des portions homogènes :

```text
1000–1250 → {forte_pente}
1250–1500 → {forte_pente, lacets}
1500–1750 → {lacets}
```

In [ ]:
import pandas as pd
from dataclasses import dataclass, asdict

## 1. Format minimal d'une annotation

In [ ]:
@dataclass
class Annotation:
    feature: str
    start_m: float
    end_m: float
    resolution_m: float
    confidence: float
    metrics: dict

## 2. Format d'une portion

In [ ]:
@dataclass
class Portion:
    start_m: float
    end_m: float
    annotations: list[str]
    details: list[dict]

## 3. Agrégation par bornes

Principe :
1. collecter tous les débuts et fins ;
2. trier ces bornes ;
3. examiner chaque intervalle ;
4. déterminer quelles annotations couvrent cet intervalle ;
5. fusionner les intervalles adjacents identiques.

In [ ]:
def build_portions(annotations):
    if not annotations:
        return []

    boundaries = sorted({
        float(a.start_m) for a in annotations
    } | {
        float(a.end_m) for a in annotations
    })

    portions = []

    for start, end in zip(boundaries[:-1], boundaries[1:]):
        if end <= start:
            continue

        active = [
            a for a in annotations
            if a.start_m <= start and a.end_m >= end
        ]

        if not active:
            continue

        feature_names = sorted({a.feature for a in active})
        details = [asdict(a) for a in active]

        current = Portion(
            start_m=start,
            end_m=end,
            annotations=feature_names,
            details=details,
        )

        # Fusion de portions adjacentes avec le même ensemble d'annotations
        if (
            portions
            and portions[-1].end_m == current.start_m
            and portions[-1].annotations == current.annotations
        ):
            portions[-1].end_m = current.end_m
            portions[-1].details.extend(current.details)
        else:
            portions.append(current)

    return portions

## 4. Exemple reproductible

In [ ]:
annotations = [
    Annotation(
        feature="forte_pente",
        start_m=1000,
        end_m=1500,
        resolution_m=25,
        confidence=0.9,
        metrics={"mean_slope_pct": 15.5},
    ),
    Annotation(
        feature="lacets",
        start_m=1250,
        end_m=1750,
        resolution_m=150,
        confidence=0.85,
        metrics={"direction_changes": 4},
    ),
]

portions = build_portions(annotations)

for p in portions:
    print(p)

## 5. Sortie attendue

```text
1000–1250 → forte_pente
1250–1500 → forte_pente + lacets
1500–1750 → lacets
```

Cette représentation devient l'entrée de l'Agent 3.